# Capstone Topic 1: System Design

> **Phase 7 → Week 16 (Capstone) → Topic 1**

---

## The Capstone Project

You are the lead data engineer at **ShopStream** — a mid-size e-commerce company processing 500K orders/day. You must design and build a complete production data platform from scratch.

### Business Requirements

| Requirement | Details |
|-------------|--------|
| Freshness SLA | Gold revenue table updated within 2 hours of event |
| Real-time dashboard | Order counts + revenue by region updated every 5 minutes |
| Historical analysis | 3 years of order history queryable by analysts via SQL |
| Data quality | < 0.1% null rate on critical columns; alerts on violations |
| Compliance | GDPR: user data deletable within 48 hours of request |
| Cost | < $5,000/month total AWS spend |

---

## Interview Cheat Sheet

**Q: Walk me through designing a real-time analytics platform for an e-commerce company.**
> Start with sources: RDS (orders), Kafka (real-time events), vendor APIs. Bronze layer ingests raw via DMS/Kafka Connect to S3. Silver layer on EMR (daily batch) validates, deduplicates, enriches. For real-time: Spark Structured Streaming reads Kafka → writes to Delta streaming table → Athena/Redshift queries it. Gold layer aggregates by date/region. Airflow orchestrates batch. CloudWatch + custom DQ checks monitor. GitHub Actions deploys code. Delta Lake ensures ACID + time travel. Total cost: EMR Spot (~$1500/mo) + S3 (~$500/mo) + Airflow MWAA (~$500/mo) + Athena (~$200/mo) = ~$2700/mo.

**Q: How do you handle a GDPR deletion request in a data lake?**
> Three steps: (1) Identify: query a `deletion_requests` table for the user's ID. (2) Delete: run `DELETE FROM delta_table WHERE user_id = X` on all tables containing PII (Bronze, Silver, Gold, dead-letter). (3) Vacuum: reduce `deletedFileRetentionDuration` and run `VACUUM` to physically remove Parquet files. Bronze is append-only by design but Delta still supports DELETE. For streaming: maintain a blocklist — new events for deleted users are filtered at Silver. SLA: complete within 48 hours. Audit trail: log every deletion with timestamp and table list.

In [ ]:
print("""
ShopStream Data Platform — Architecture Decision Record
══════════════════════════════════════════════════════════════

SOURCES
  orders_db     → PostgreSQL RDS (transactional, 500K orders/day)
  events_kafka  → Kafka MSK (clickstream, ~5M events/day)
  vendor_sftp   → Supplier inventory files (hourly CSV drops)

INGEST
  RDS → DMS (CDC) → S3 Bronze (Parquet, date-partitioned)
  Kafka → Spark Structured Streaming → S3 Bronze + Delta streaming table
  SFTP → Lambda (triggered by S3 event) → S3 Bronze

STORAGE
  Format:      Delta Lake (ACID, time travel, MERGE, CDF)
  Layout:      s3://shopstream-data/
                 bronze/{source}/{year}/{month}/{day}/
                 silver/{domain}/{date=YYYY-MM-DD}/
                 gold/{metric}/{date=YYYY-MM-DD}/
                 streaming/{table}/  (checkpoint + data)
                 dead-letter/{layer}/{date=YYYY-MM-DD}/

BATCH PROCESSING (daily, 6am UTC)
  Orchestrator:  Apache Airflow (MWAA)
  Compute:       EMR 6.15 — m5.xlarge master, m5.2xlarge core×2, m5.2xlarge task×8 (Spot)
  Pipeline:      Bronze validate → Silver transform+DQ → Gold aggregate → freshness check
  Idempotency:   Dynamic partition overwrite + Delta MERGE

STREAMING (continuous)
  Compute:       Spark Structured Streaming on EMR (long-running cluster)
  Source:        Kafka MSK (orders topic)
  Sink:          Delta streaming table (5-min trigger)
  Watermark:     10 minutes for late events
  Checkpoint:    s3://shopstream-data/checkpoints/

SERVING
  Analyst SQL:   Athena → Glue Catalog → S3 Gold (pay-per-query, no ETL)
  BI Dashboard:  QuickSight → SPICE (5-min refresh from streaming Delta table)
  Data Science:  SageMaker → reads Silver Delta directly

OBSERVABILITY
  Airflow UI:    DAG run history, task logs, SLA tracking
  CloudWatch:    EMR metrics, custom DQ metrics, freshness lag
  PagerDuty:     P1 alerts (Gold not updated, row count drop >50%)
  Slack:         P2/P3 alerts (DQ violations, single job failure)

GOVERNANCE
  Catalog:       Glue Data Catalog (schema registry, partition metadata)
  Access:        Lake Formation (column-level for PII, row-level for region)
  GDPR:          Deletion pipeline (Delta DELETE + VACUUM within 48h)
  Schema:        Glue Schema Registry (BACKWARD compatible, Avro)

CI/CD
  GitHub Actions: lint (ruff) → test (pytest, 80% coverage) → deploy to S3
  OIDC:          No static AWS credentials in CI
  Versioning:    Code uploaded to s3://code/{git-sha}/libs.zip

COST ESTIMATE (monthly)
  EMR batch  (3h/day × 11 nodes × $0.10/node-hr avg with Spot): ~$1,000
  EMR stream (24/7 × 3 nodes × $0.13/node-hr):                  ~$280
  S3 (2 TB data + requests):                                     ~$500
  MWAA (Airflow):                                                 ~$500
  Athena (100 GB scanned/day × $5/TB):                           ~$150
  MSK Kafka (3 brokers):                                          ~$450
  Other (Lambda, CloudWatch, QuickSight):                         ~$200
  TOTAL:                                                         ~$3,080/month ✅
══════════════════════════════════════════════════════════════
""")

## Technology Decision Matrix

In [ ]:
print("""
Key Technology Decisions and Justifications:
══════════════════════════════════════════════════════════════

Decision 1: Delta Lake vs Iceberg vs Hudi
  Choice:  Delta Lake
  Reason:  Primary compute is Spark on EMR. Delta has best Spark integration,
           MERGE INTO is well-tested, OPTIMIZE + ZORDER for data skipping.
           CDF for GDPR deletion propagation.
  Tradeoff: Iceberg would be better if we needed Flink or Trino in future.

Decision 2: EMR vs Glue vs Databricks
  Batch:   EMR (cost control with Spot, custom Spark configs, Spark UI)
  Reason:  500K orders/day = ~4 GB/day. Job runs 3h/day → cluster idle 21h.
           Spot saves 60-80% vs On-Demand. Glue would cost 3× more.
           Databricks adds DBU cost on top of EC2 — not worth it at this scale.
  Tradeoff: Databricks would be chosen if team doubles and collaboration matters more.

Decision 3: Airflow vs Step Functions vs Databricks Workflows
  Choice:  Airflow (MWAA)
  Reason:  Team already knows Airflow. Need cross-system dependencies
           (wait for SFTP file AND Kafka lag = 0 before running Gold).
           Step Functions would work but less expressive for complex DAGs.
  Tradeoff: MWAA adds $500/month vs self-managed Airflow on ECS.

Decision 4: Athena vs Redshift for serving
  Choice:  Athena (primary) + Redshift Spectrum (heavy BI)
  Reason:  Athena: zero infra, pay-per-query, immediate for ad-hoc.
           Redshift: for QuickSight dashboards with complex JOINs that scan
           full Gold tables repeatedly (SPICE can cache but still needs RS for refresh).
  Tradeoff: Athena at 100 GB/day → $150/month. Acceptable.

Decision 5: Streaming — Spark Structured Streaming vs Flink
  Choice:  Spark Structured Streaming
  Reason:  Team already knows Spark. Reuse EMR cluster. Delta sink is first-class.
           5-minute latency SLA → micro-batch is fine (no need for Flink's ms latency).
  Tradeoff: Flink would be chosen if latency < 1 second was required.
══════════════════════════════════════════════════════════════
""")

## Exercises

1. **Design review**: A colleague proposes using AWS Glue instead of EMR for the batch pipeline. Walk through the cost comparison and list three technical reasons why EMR is the better choice for this specific workload.
2. **Capacity planning**: The business expects 3× growth in 18 months (1.5M orders/day, 15M events/day). How would the architecture change? Which components would you scale first and how?
3. **GDPR design**: Design the full deletion pipeline for ShopStream. Which tables contain PII? What is the exact sequence of Delta DELETE + VACUUM operations? How do you audit that deletion was complete?
4. **Cost optimization**: Identify three changes to the architecture that would reduce the monthly cost from $3,080 while keeping the same SLAs. Estimate the savings for each.
5. **Failure mode analysis**: What happens if the Kafka MSK cluster is unavailable for 2 hours? Trace the failure through every system layer and describe the recovery procedure.